# Claude API Usage Analysis

This notebook analyzes the API token usage and costs for the PyDataVT2025 project development using Claude Code.

## Overview

This analysis visualizes:
1. **Token Usage** - Input and output tokens by date and model
2. **API Costs** - Costs broken down by token type and date

The data demonstrates the cost and efficiency of using AI-assisted development tools.

In [ ]:
# Import required libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter

print("✓ Libraries imported successfully")

## 1. API Token Usage Analysis

This section analyzes token consumption patterns across different models and cache strategies.

### Token Types Explained:
- **input_no_cache**: Fresh input tokens without caching
- **input_cache_write_5m**: Input tokens written to 5-minute cache
- **input_cache_read**: Input tokens read from cache (cheaper than fresh input)
- **output**: Generated response tokens

In [ ]:
# Load token usage data
tokens_df = pd.read_csv('claude_api_tokens.csv')

# Convert date column to datetime
tokens_df['usage_date_utc'] = pd.to_datetime(tokens_df['usage_date_utc'])

# Display raw data
print("Raw Token Usage Data:")
print(tokens_df.to_string(index=False))
print(f"\nTotal rows: {len(tokens_df)}")

# Calculate total tokens by type
print("\n" + "="*60)
print("TOKEN USAGE SUMMARY")
print("="*60)

total_input_no_cache = tokens_df['usage_input_tokens_no_cache'].sum()
total_cache_write = tokens_df['usage_input_tokens_cache_write_5m'].sum()
total_cache_read = tokens_df['usage_input_tokens_cache_read'].sum()
total_output = tokens_df['usage_output_tokens'].sum()
total_all_tokens = total_input_no_cache + total_cache_write + total_cache_read + total_output

print(f"\nInput Tokens (No Cache):     {total_input_no_cache:>12,}")
print(f"Input Tokens (Cache Write):  {total_cache_write:>12,}")
print(f"Input Tokens (Cache Read):   {total_cache_read:>12,}")
print(f"Output Tokens:               {total_output:>12,}")
print(f"{'-'*60}")
print(f"TOTAL TOKENS:                {total_all_tokens:>12,}")

# Calculate by model
print("\n" + "="*60)
print("TOKENS BY MODEL")
print("="*60)
for model in tokens_df['model_version'].unique():
    model_data = tokens_df[tokens_df['model_version'] == model]
    model_total = (model_data['usage_input_tokens_no_cache'].sum() + 
                   model_data['usage_input_tokens_cache_write_5m'].sum() + 
                   model_data['usage_input_tokens_cache_read'].sum() + 
                   model_data['usage_output_tokens'].sum())
    print(f"\n{model}:")
    print(f"  Total: {model_total:,} tokens")
    print(f"  % of total: {model_total/total_all_tokens*100:.1f}%")

In [ ]:
# Create comprehensive token visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Claude API Token Usage Analysis', fontsize=16, fontweight='bold')

# Helper function to format large numbers
def millions(x, pos):
    return f'{x*1e-6:.1f}M'

# 1. Stacked bar chart by date
ax1 = axes[0, 0]
date_grouped = tokens_df.groupby('usage_date_utc').sum(numeric_only=True)
x_dates = np.arange(len(date_grouped))
width = 0.6

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
bottom = np.zeros(len(date_grouped))

for i, (col, label, color) in enumerate([
    ('usage_input_tokens_no_cache', 'Input (No Cache)', colors[0]),
    ('usage_input_tokens_cache_write_5m', 'Input (Cache Write)', colors[1]),
    ('usage_input_tokens_cache_read', 'Input (Cache Read)', colors[2]),
    ('usage_output_tokens', 'Output', colors[3])
]):
    ax1.bar(x_dates, date_grouped[col], width, label=label, bottom=bottom, color=color)
    bottom += date_grouped[col]

ax1.set_xlabel('Date', fontweight='bold')
ax1.set_ylabel('Tokens', fontweight='bold')
ax1.set_title('Token Usage Over Time (Stacked)', fontweight='bold')
ax1.set_xticks(x_dates)
ax1.set_xticklabels([d.strftime('%m/%d') for d in date_grouped.index], rotation=45)
ax1.legend(loc='upper left')
ax1.yaxis.set_major_formatter(FuncFormatter(millions))
ax1.grid(axis='y', alpha=0.3)

# 2. Pie chart of token types
ax2 = axes[0, 1]
token_totals = [total_input_no_cache, total_cache_write, total_cache_read, total_output]
token_labels = ['Input\n(No Cache)', 'Input\n(Cache Write)', 'Input\n(Cache Read)', 'Output']
wedges, texts, autotexts = ax2.pie(token_totals, labels=token_labels, autopct='%1.1f%%',
                                     colors=colors, startangle=90)
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
ax2.set_title('Token Distribution by Type', fontweight='bold')

# 3. Token usage by model
ax3 = axes[1, 0]
model_grouped = tokens_df.groupby('model_version').sum(numeric_only=True)
model_totals = (model_grouped['usage_input_tokens_no_cache'] + 
                model_grouped['usage_input_tokens_cache_write_5m'] + 
                model_grouped['usage_input_tokens_cache_read'] + 
                model_grouped['usage_output_tokens'])

model_names = ['Haiku 4.5', 'Sonnet 4.5']  # Shortened names for display
bars = ax3.barh(model_names, model_totals.values, color=['#8B4513', '#4169E1'])
ax3.set_xlabel('Total Tokens', fontweight='bold')
ax3.set_title('Total Token Usage by Model', fontweight='bold')
ax3.xaxis.set_major_formatter(FuncFormatter(millions))
ax3.grid(axis='x', alpha=0.3)

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, model_totals.values)):
    ax3.text(val, bar.get_y() + bar.get_height()/2, f'{val/1e6:.2f}M',
             ha='left', va='center', fontweight='bold', fontsize=10, color='black')

# 4. Cache efficiency analysis
ax4 = axes[1, 1]
cache_data = tokens_df.groupby('usage_date_utc')[['usage_input_tokens_cache_write_5m', 
                                                    'usage_input_tokens_cache_read']].sum()
x_dates = np.arange(len(cache_data))
width = 0.35

bars1 = ax4.bar(x_dates - width/2, cache_data['usage_input_tokens_cache_write_5m'], 
                width, label='Cache Write', color='#ff7f0e')
bars2 = ax4.bar(x_dates + width/2, cache_data['usage_input_tokens_cache_read'], 
                width, label='Cache Read', color='#2ca02c')

ax4.set_xlabel('Date', fontweight='bold')
ax4.set_ylabel('Tokens', fontweight='bold')
ax4.set_title('Cache Usage Efficiency', fontweight='bold')
ax4.set_xticks(x_dates)
ax4.set_xticklabels([d.strftime('%m/%d') for d in cache_data.index], rotation=45)
ax4.legend()
ax4.yaxis.set_major_formatter(FuncFormatter(millions))
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Token usage visualization complete")

## 2. API Cost Analysis

This section analyzes the monetary costs of API usage.

### Cost Structure:
- Different token types have different costs
- Cache reads are typically cheaper than fresh inputs
- Output tokens are generally more expensive than input tokens
- Sonnet models cost more than Haiku models but provide higher quality responses

In [ ]:
# Load cost data
cost_df = pd.read_csv('claude_api_cost.csv')

# Convert date column to datetime
cost_df['usage_date_utc'] = pd.to_datetime(cost_df['usage_date_utc'])

# Display raw data
print("Raw Cost Data:")
print(cost_df.to_string(index=False))
print(f"\nTotal rows: {len(cost_df)}")

# Calculate total costs
print("\n" + "="*60)
print("COST SUMMARY")
print("="*60)

total_cost = cost_df['cost_usd'].sum()
print(f"\nTotal API Cost: ${total_cost:.2f}")

# Cost by model
print("\n" + "="*60)
print("COST BY MODEL")
print("="*60)
for model in cost_df['model'].unique():
    model_cost = cost_df[cost_df['model'] == model]['cost_usd'].sum()
    print(f"\n{model}:")
    print(f"  Total: ${model_cost:.2f}")
    print(f"  % of total: {model_cost/total_cost*100:.1f}%")

# Cost by token type
print("\n" + "="*60)
print("COST BY TOKEN TYPE")
print("="*60)
for token_type in cost_df['token_type'].unique():
    type_cost = cost_df[cost_df['token_type'] == token_type]['cost_usd'].sum()
    print(f"\n{token_type}:")
    print(f"  Total: ${type_cost:.2f}")
    print(f"  % of total: {type_cost/total_cost*100:.1f}%")

# Daily costs
print("\n" + "="*60)
print("DAILY COSTS")
print("="*60)
daily_costs = cost_df.groupby('usage_date_utc')['cost_usd'].sum()
for date, cost in daily_costs.items():
    print(f"\n{date.strftime('%m/%d/%Y')}: ${cost:.2f}")

In [ ]:
# Create comprehensive cost visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Claude API Cost Analysis', fontsize=16, fontweight='bold')

# 1. Daily costs stacked by model
ax1 = axes[0, 0]
daily_model_costs = cost_df.groupby(['usage_date_utc', 'model'])['cost_usd'].sum().unstack(fill_value=0)
daily_model_costs.plot(kind='bar', stacked=True, ax=ax1, color=['#8B4513', '#4169E1'])
ax1.set_xlabel('Date', fontweight='bold')
ax1.set_ylabel('Cost (USD)', fontweight='bold')
ax1.set_title('Daily Costs by Model (Stacked)', fontweight='bold')
ax1.set_xticklabels([d.strftime('%m/%d') for d in daily_model_costs.index], rotation=45)
ax1.legend(title='Model', labels=['Haiku 4.5', 'Sonnet 4.5'])
ax1.grid(axis='y', alpha=0.3)

# Add total cost labels on bars
daily_totals = daily_model_costs.sum(axis=1)
for i, (idx, total) in enumerate(daily_totals.items()):
    ax1.text(i, total, f'${total:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

# 2. Pie chart of total cost by model
ax2 = axes[0, 1]
model_costs = cost_df.groupby('model')['cost_usd'].sum()
wedges, texts, autotexts = ax2.pie(model_costs.values, labels=['Haiku 4.5', 'Sonnet 4.5'],
                                     autopct=lambda pct: f'${pct*total_cost/100:.2f}\n({pct:.1f}%)',
                                     colors=['#8B4513', '#4169E1'], startangle=90)
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
ax2.set_title(f'Total Cost by Model\n(${total_cost:.2f} total)', fontweight='bold')

# 3. Cost by token type
ax3 = axes[1, 0]
token_type_costs = cost_df.groupby('token_type')['cost_usd'].sum().sort_values(ascending=True)
colors_token = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'][:len(token_type_costs)]
bars = ax3.barh(range(len(token_type_costs)), token_type_costs.values, color=colors_token)
ax3.set_yticks(range(len(token_type_costs)))
ax3.set_yticklabels([label.replace('input_', '').replace('_', '\n') for label in token_type_costs.index])
ax3.set_xlabel('Cost (USD)', fontweight='bold')
ax3.set_title('Cost Breakdown by Token Type', fontweight='bold')
ax3.grid(axis='x', alpha=0.3)

# Add value labels
for i, (bar, val) in enumerate(zip(bars, token_type_costs.values)):
    ax3.text(val, bar.get_y() + bar.get_height()/2, f'  ${val:.2f}',
             ha='left', va='center', fontweight='bold', fontsize=9)

# 4. Cumulative cost over time
ax4 = axes[1, 1]
daily_totals = cost_df.groupby('usage_date_utc')['cost_usd'].sum().sort_index()
cumulative_costs = daily_totals.cumsum()
ax4.plot(range(len(cumulative_costs)), cumulative_costs.values, marker='o', 
         linewidth=2, markersize=8, color='#2ca02c')
ax4.fill_between(range(len(cumulative_costs)), cumulative_costs.values, alpha=0.3, color='#2ca02c')
ax4.set_xlabel('Date', fontweight='bold')
ax4.set_ylabel('Cumulative Cost (USD)', fontweight='bold')
ax4.set_title('Cumulative API Costs Over Time', fontweight='bold')
ax4.set_xticks(range(len(cumulative_costs)))
ax4.set_xticklabels([d.strftime('%m/%d') for d in cumulative_costs.index], rotation=45)
ax4.grid(True, alpha=0.3)

# Add value labels on points
for i, (date, cost) in enumerate(cumulative_costs.items()):
    ax4.text(i, cost, f'${cost:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✓ Cost analysis visualization complete")

## Key Insights

### Token Usage Patterns:
- Cache reads significantly reduce costs by reusing context
- Sonnet 4.5 handles the majority of tokens due to its role in primary development tasks
- Haiku 4.5 is used for lighter tasks with smaller token counts

### Cost Efficiency:
- Cache strategies (5-minute cache) provide substantial cost savings
- The cumulative cost over the project timeline shows the total investment in AI-assisted development
- Different token types have different pricing, with cache reads being more economical

### Development Insights:
- This data demonstrates transparency in AI-assisted development costs
- The costs are reasonable for the productivity gains and learning outcomes achieved
- Caching strategies are essential for cost-effective use of Claude Code